In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "xguman/hw07_finetuned_slovak_books"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# GPT-style models often lack a pad token — use EOS as fallback
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pipeline approach
# text_gen_pipeline = pipeline(
#     "text-generation",
#     model=model_name,
#     tokenizer=model_name,
#     device=0 if torch.cuda.is_available() else -1
# )

# print("\n--- Text Generation Example (Pipeline Way) ---")

# prompt_pipeline_1 = "Bol raz jeden starý zámok"
# result_pipeline_1 = text_gen_pipeline(prompt_pipeline_1, max_new_tokens=60, do_sample=True, temperature=0.8)
# print(f"\nPrompt: '{prompt_pipeline_1}'")
# print(f"Generated: {result_pipeline_1[0]['generated_text']}")

# prompt_pipeline_2 = "Vo vysokých horách žil zbojník"
# result_pipeline_2 = text_gen_pipeline(prompt_pipeline_2, max_new_tokens=60, do_sample=True, temperature=0.8)
# print(f"\nPrompt: '{prompt_pipeline_2}'")
# print(f"Generated: {result_pipeline_2[0]['generated_text']}")

print("\n--- Manual Text Generation Example ---")

def generate_text_manual(
    prompt,
    model,
    tokenizer,
    device,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
):
    # 1. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        max_length=512,
    ).to(device)

    prompt_length = inputs["input_ids"].shape[1]

    # 2. Generate tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 3. Decode only the newly generated tokens
    generated_ids = output_ids[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "prompt": prompt,
        "generated": generated_text,
        "full_text": prompt + generated_text,
    }

# Example 1: Book/fairy-tale style opening (fits the model's training domain)
prompt_1 = "Bol raz jeden starý zámok, v ktorom"
result_1 = generate_text_manual(prompt_1, model, tokenizer, device)
print(f"\nPrompt: '{result_1['prompt']}'")
print(f"Generated continuation: '{result_1['generated']}'")
print(f"Full text: '{result_1['full_text']}'")

# Example 2: Another literary/narrative prompt
prompt_2 = "Vo vysokých horách žil zbojník, ktorý"
result_2 = generate_text_manual(prompt_2, model, tokenizer, device)
print(f"\nPrompt: '{result_2['prompt']}'")
print(f"Generated continuation: '{result_2['generated']}'")
print(f"Full text: '{result_2['full_text']}'")

Loading xguman/hw07_finetuned_slovak_books...


config.json:   0%|          | 0.00/926 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Manual Text Generation Example ---

Prompt: 'Bol raz jeden starý zámok, v ktorom'
Generated continuation: ' neexistuje život. Staré a hlučné krovy sa vyťahujú až do miesta smrti... Všetky dve strany sú preplnené bojom za presunutie, ale v stene je miesto bezpečia - tam môže ostať človek s myšlienkou o úteku. Môže to byť obzvlášť podnecovaný ch'
Full text: 'Bol raz jeden starý zámok, v ktorom neexistuje život. Staré a hlučné krovy sa vyťahujú až do miesta smrti... Všetky dve strany sú preplnené bojom za presunutie, ale v stene je miesto bezpečia - tam môže ostať človek s myšlienkou o úteku. Môže to byť obzvlášť podnecovaný ch'

Prompt: 'Vo vysokých horách žil zbojník, ktorý'
Generated continuation: ' sa dostal do konfliktu s policiou. Svojou kriminalitou si našiel vďaka aj úspechy na ceste k svätému poslednému prikrmu.

<h2>Vokaličár v „dobrom“ vo väzení</h2>

<strong>„Pridáte ma? Hľadem nahor.“</strong> V tomto prípade je „nahorský“,'
Full text: 'Vo vysokých horách žil zbojník, kt

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
